# VoxSync Colab (online)

Transcribe video o YouTube, traduce a espanol **sin Google** (MarianMT local) y genera:
- SRT en espanol
- Video MP4 con subtitulos quemados (voces originales + texto en ES)

1. Entorno de ejecucion > Cambiar tipo de entorno > GPU (T4)
2. Ejecuta las celdas en orden
3. Descarga el SRT y/o el video al final


## 1. Instalar dependencias


In [ ]:
!pip install -q openai-whisper yt-dlp "transformers>=4.40,<5" sentencepiece sacremoses
!apt -y -qq install ffmpeg
print("OK")


## 2. Configuracion


In [ ]:
YOUTUBE_URL = ""  # ejemplo: "https://www.youtube.com/watch?v=XXXX"
SOURCE_LANG = "en"
TARGET_LANG = "es"
WHISPER_MODEL = "small"  # tiny | base | small | medium
BURN_SUBTITLES = True
print("Config OK")


## 3. Cargar video


In [ ]:
from pathlib import Path
import subprocess

work = Path("/content/voxsync")
work.mkdir(exist_ok=True)

if YOUTUBE_URL.strip():
    print("Descargando YouTube...")
    subprocess.run([
        "yt-dlp", "-f", "bv*[ext=mp4]+ba[ext=m4a]/b[ext=mp4]/b",
        "-o", str(work / "%(title)s.%(ext)s"), "--no-playlist", YOUTUBE_URL.strip()
    ], check=True)
    cands = []
    for ext in (".mp4", ".webm", ".mkv", ".m4a", ".mp3"):
        cands.extend(work.glob("*" + ext))
    video_path = max(cands, key=lambda p: p.stat().st_mtime)
    print("Video:", video_path.name)
else:
    from google.colab import files
    print("Sube un video o audio...")
    uploaded = files.upload()
    name = list(uploaded.keys())[0]
    video_path = work / name
    video_path.write_bytes(uploaded[name])
    print("Archivo:", video_path.name)

print("Listo:", video_path)


## 4. Extraer audio


In [ ]:
audio_path = work / "audio.wav"
subprocess.run([
    "ffmpeg", "-y", "-i", str(video_path),
    "-vn", "-acodec", "pcm_s16le", "-ar", "16000", "-ac", "1",
    str(audio_path)
], check=True, capture_output=True)
print("Audio OK:", audio_path)


## 5. Transcribir con Whisper


In [ ]:
import whisper

print("Cargando", WHISPER_MODEL)
model = whisper.load_model(WHISPER_MODEL)
print("Transcribiendo...")
result = model.transcribe(str(audio_path), language=SOURCE_LANG or None, verbose=False)
segments = result["segments"]
print(len(segments), "segmentos")
for s in segments[:3]:
    print(round(s["start"], 1), s["text"][:100])


## 6. Traducir LOCAL con MarianMT (compatible transformers v4 y v5)


In [ ]:
import torch
from transformers import MarianMTModel, MarianTokenizer

model_name = "Helsinki-NLP/opus-mt-en-es"
if SOURCE_LANG == "es" and TARGET_LANG == "en":
    model_name = "Helsinki-NLP/opus-mt-es-en"

print("Cargando", model_name)
tokenizer = MarianTokenizer.from_pretrained(model_name)
mt_model = MarianMTModel.from_pretrained(model_name)
device = "cuda" if torch.cuda.is_available() else "cpu"
mt_model = mt_model.to(device)
mt_model.eval()
print("Dispositivo:", device)

def translate_batch(texts, max_length=256):
    texts = [t if (t and t.strip()) else "." for t in texts]
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_length,
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = mt_model.generate(**inputs, max_length=max_length)
    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

texts = [(s.get("text") or "").strip() for s in segments]
translations = []
batch_size = 16

if SOURCE_LANG == TARGET_LANG:
    translations = list(texts)
else:
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        out = translate_batch(batch)
        for j, t in enumerate(batch):
            translations.append(out[j] if t else "")
        print(f"  traducidos {min(i + batch_size, len(texts))}/{len(texts)}")

for seg, tr in zip(segments, translations):
    seg["translation"] = tr

print("OK - vista previa:")
for s in segments[:3]:
    print("ORIG:", s["text"][:80])
    print("TRAD:", s["translation"][:80])
    print()


## 7. Generar SRT


In [ ]:
def ts(seconds):
    total = int(float(seconds))
    ms = int((float(seconds) - total) * 1000)
    h, rem = divmod(total, 3600)
    m, s = divmod(rem, 60)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

base = video_path.stem
srt_path = work / f"{base}.{TARGET_LANG}.srt"
srt_bi = work / f"{base}.bilingual.srt"

with open(srt_path, "w", encoding="utf-8") as f:
    for i, seg in enumerate(segments, 1):
        text = (seg.get("translation") or seg["text"]).strip()
        f.write(f"{i}\n{ts(seg['start'])} --> {ts(seg['end'])}\n{text}\n\n")

with open(srt_bi, "w", encoding="utf-8") as f:
    for i, seg in enumerate(segments, 1):
        orig = seg["text"].strip()
        trans = (seg.get("translation") or orig).strip()
        body = orig + "\n" + trans if trans != orig else orig
        f.write(f"{i}\n{ts(seg['start'])} --> {ts(seg['end'])}\n{body}\n\n")

print("SRT:", srt_path)
print("Bilingual:", srt_bi)


## 8. Video con subtitulos quemados


In [ ]:
out_video = None
if BURN_SUBTITLES:
    out_video = work / f"{base}_ES.mp4"
    srt_for_ffmpeg = str(srt_path).replace("\\", "/").replace(":", "\\:")
    print("Generando video con subtitulos...")
    subprocess.run([
        "ffmpeg", "-y", "-i", str(video_path),
        "-vf", f"subtitles={srt_for_ffmpeg}:force_style='FontSize=18,Outline=1'",
        "-c:a", "copy",
        str(out_video)
    ], check=True)
    print("Video listo:", out_video)
else:
    print("BURN_SUBTITLES=False: solo SRT")


## 9. Descargar resultados


In [ ]:
from google.colab import files

files.download(str(srt_path))
files.download(str(srt_bi))
if out_video is not None and Path(out_video).exists():
    files.download(str(out_video))
print("Listo: SRT en espanol + video con voces y subtitulos")
